# Scientific Figure Layout Designer for Publication-Quality Plots

Hello everyone,

In today's video, I am going to show you a Python project called Scientific Figure Layout Designer .

This tool helps us create custom layouts for scientific figures using Matplotlib.


# Code link in the description #    

When creating scientific plots, we often need multiple graphs in one figure. For example, we may need a large main plot, smaller supporting plots, polar plots, or 3D visualizations. Managing the position and size of each subplot manually can become difficult.

So, the goal of this project is to create a visual editor where we can design the figure layout interactively.

The user can select different types of axes:

* Cartesian axes
* Polar axes
* 3D axes
* Inset figures

Lets run the code and see there. 



This makes designing complex scientific figures much faster and easier.

The project is built using Python, Matplotlib, and interactive event handling.

So this is the basic idea behind the Scientific Figure Layout Designer.


Thank you for watching, and I will see you in the next video.



In [ ]:
%matplotlib qt

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import Button
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.patches import Circle
import numpy as np

xsine = np.linspace(0, 2*np.pi, 100)
ysine = np.sin(xsine)


corner_handles = []
HANDLE_RADIUS = 1.2
active_handle = None

# ==========================================================
# GRID SETTINGS
# ==========================================================

GRID_ROWS = 100
GRID_COLS = 100

projection_mode = "rectilinear"

delete_mode = False
resize_mode = False

selected_idx = None

# resize information
resize_corner = None
HANDLE_SIZE = 2      # distance (grid cells) for detecting corner

projection_colors = {
    "rectilinear": "red",
    "polar": "royalblue",
    "3d": "forestgreen",
}

rectangles = []

start = None
current_rect = None

# ==========================================================
# FIGURE
# ==========================================================

fig, ax = plt.subplots(figsize=(10,10))
plt.subplots_adjust(bottom=0.20)

ax.set_xlim(0, GRID_COLS)
ax.set_ylim(GRID_ROWS, 0)
ax.set_aspect("equal")

ax.set_xticks(range(0, GRID_COLS + 1, 5))
ax.set_yticks(range(0, GRID_ROWS + 1, 5))

ax.set_xticks(range(GRID_COLS + 1), minor=True)
ax.set_yticks(range(GRID_ROWS + 1), minor=True)

ax.grid(which="minor", color="lightgray", linewidth=0.5)
ax.grid(which="major", color="gray", linewidth=1)

# ==========================================================
# BUTTONS
# ==========================================================

b1 = plt.axes([0.05,0.03,0.14,0.05])
b2 = plt.axes([0.20,0.03,0.14,0.05])
b3 = plt.axes([0.35,0.03,0.14,0.05])
bd = plt.axes([0.50,0.03,0.14,0.05])
br = plt.axes([0.65,0.03,0.14,0.05])
bp = plt.axes([0.80,0.03,0.14,0.05])
be = plt.axes([0.05,0.10,0.14,0.05])

btn_cart = Button(b1,"Cartesian")
btn_polar = Button(b2,"Polar")
btn_3d = Button(b3,"3D")
btn_delete = Button(bd,"Delete")
btn_resize = Button(br,"Resize")
btn_preview = Button(bp,"Preview")
btn_export = Button(be,"Export Code")

status = fig.text(
    0.02,
    0.96,
    "Mode: Cartesian",
    fontsize=11,
    weight="bold"
)

# ==========================================================
# BUTTON COLORS
# ==========================================================

def set_active(btn, active=True):
    btn.ax.set_facecolor(
        "lightgreen" if active else "lightgray"
    )
    fig.canvas.draw_idle()

def find_corner(rect, x, y):
    """
    Return which corner of the rectangle is being clicked.

    Returns:
        "tl", "tr", "bl", "br" or None
    """

    x0 = rect["col"]
    y0 = rect["row"]
    x1 = x0 + rect["colspan"]
    y1 = y0 + rect["rowspan"]

    corners = {
        "tl": (x0, y0),
        "tr": (x1, y0),
        "bl": (x0, y1),
        "br": (x1, y1),
    }

    nearest = None
    min_dist = float("inf")

    for name, (cx, cy) in corners.items():
        dist = ((x - cx) ** 2 + (y - cy) ** 2) ** 0.5

        if dist < HANDLE_SIZE and dist < min_dist:
            nearest = name
            min_dist = dist

    return nearest

def reset_buttons():
    for b in [
        btn_cart,
        btn_polar,
        btn_3d,
        btn_delete,
        btn_resize,
    ]:
        b.ax.set_facecolor("lightgray")

# ==========================================================
# MODES
# ==========================================================

def set_mode(mode):

    global projection_mode
    global delete_mode
    global resize_mode

    projection_mode = mode
    delete_mode = False
    resize_mode = False

    reset_buttons()

    if mode == "rectilinear":
        set_active(btn_cart)

    elif mode == "polar":
        set_active(btn_polar)

    elif mode == "3d":
        set_active(btn_3d)

    status.set_text(f"Mode: {mode}")
    fig.canvas.draw_idle()


btn_cart.on_clicked(lambda e: set_mode("rectilinear"))
btn_polar.on_clicked(lambda e: set_mode("polar"))
btn_3d.on_clicked(lambda e: set_mode("3d"))

# ==========================================================
# DELETE MODE
# ==========================================================

def toggle_delete(event):

    global delete_mode
    global resize_mode

    delete_mode = not delete_mode
    resize_mode = False

    reset_buttons()

    set_active(btn_delete, delete_mode)

    if delete_mode:
        status.set_text("Mode: DELETE")
    else:
        status.set_text("Mode: DRAW")

    fig.canvas.draw_idle()


btn_delete.on_clicked(toggle_delete)

# ==========================================================
# RESIZE MODE
# ==========================================================

def toggle_resize(event):

    global resize_mode
    global delete_mode

    resize_mode = not resize_mode
    delete_mode = False

    reset_buttons()

    set_active(btn_resize, resize_mode)

    if resize_mode:
        status.set_text("Mode: RESIZE")
    else:
        status.set_text("Mode: DRAW")

    fig.canvas.draw_idle()


btn_resize.on_clicked(toggle_resize)

def draw_handles(rect):
    global corner_handles

    # remove old handles
    for h in corner_handles:
        h.remove()
    corner_handles = []

    x0 = rect["col"]
    y0 = rect["row"]
    x1 = x0 + rect["colspan"]
    y1 = y0 + rect["rowspan"]

    corners = [
        (x0, y0),  # tl
        (x1, y0),  # tr
        (x0, y1),  # bl
        (x1, y1),  # br
    ]

    rect["handles"] = []

    for cx, cy in corners:
        c = Circle(
            (cx, cy),
            HANDLE_RADIUS,
            facecolor="black",
            edgecolor="white",
            linewidth=1,
            zorder=10,
            picker=True,   # ✅ clickable
        )
        ax.add_patch(c)
        corner_handles.append(c)
        rect["handles"].append(c)

    fig.canvas.draw_idle()

# ==========================================================
# HELPER FUNCTIONS
# ==========================================================

def find_rect(event):
    """
    Return index of rectangle containing mouse position.
    """
    if event.inaxes != ax:
        return None

    if event.xdata is None or event.ydata is None:
        return None

    for i, r in enumerate(rectangles):

        x0 = r["col"]
        y0 = r["row"]
        x1 = x0 + r["colspan"]
        y1 = y0 + r["rowspan"]

        if x0 <= event.xdata <= x1 and y0 <= event.ydata <= y1:
            return i

    return None


def draw_selection(idx):

    for i, r in enumerate(rectangles):

        if i == idx:
            r["patch"].set_linewidth(3)
        else:
            r["patch"].set_linewidth(1.5)

    if idx is not None:
        draw_handles(rectangles[idx])
    else:
        for h in corner_handles:
            h.remove()
        corner_handles.clear()

    fig.canvas.draw_idle()


# ==========================================================
# MOUSE PRESS
# ==========================================================

def on_press(event):

    global active_handle
    global start
    global current_rect
    global selected_idx
    global resize_corner

    # --------------------------------------------------
    # CHECK IF A HANDLE WAS CLICKED
    # --------------------------------------------------
    if resize_mode:

        for i, rect in enumerate(rectangles):
            if "handles" in rect:
                for j, h in enumerate(rect["handles"]):
                    contains, _ = h.contains(event)

                    if contains:
                        selected_idx = i
                        resize_corner = ["tl", "tr", "bl", "br"][j]
                        draw_selection(i)
                        active_handle = h
                        return

    if event.inaxes != ax:
        return

    if event.xdata is None or event.ydata is None:
        return

    # --------------------------------------------------
    # DELETE MODE
    # --------------------------------------------------

    if delete_mode:

        idx = find_rect(event)

        if idx is not None:

            r = rectangles.pop(idx)

            r["patch"].remove()
            r["label"].remove()

            fig.canvas.draw_idle()

        return

    # --------------------------------------------------
    # RESIZE MODE
    # --------------------------------------------------

    if resize_mode:

        selected_idx = find_rect(event)

        if selected_idx is None:
            return

        draw_selection(selected_idx)

        resize_corner = find_corner(
            rectangles[selected_idx],
            event.xdata,
            event.ydata,
        )

        # Only start resizing if a corner was clicked
        return

    # --------------------------------------------------
    # DRAW MODE
    # --------------------------------------------------

    start = (
        int(event.xdata),
        int(event.ydata)
    )

    current_rect = Rectangle(
        start,
        1,
        1,
        fill=False,
        linewidth=2,
        edgecolor=projection_colors[projection_mode],
    )

    ax.add_patch(current_rect)

    selected_idx = find_rect(event)
    draw_selection(selected_idx)

    fig.canvas.draw_idle()

    # ==========================================================
# MOUSE MOTION
# ==========================================================

def on_motion(event):

    global start
    global current_rect
    global selected_idx
    global resize_corner

    if event.inaxes != ax:
        return

    if event.xdata is None or event.ydata is None:
        return

    # --------------------------------------------------
    # RESIZE
    # --------------------------------------------------
    if resize_mode:

        if selected_idx is None:
            return

        r = rectangles[selected_idx]

        x0 = r["col"]
        y0 = r["row"]
        x1 = x0 + r["colspan"]
        y1 = y0 + r["rowspan"]

        mx = int(event.xdata)
        my = int(event.ydata)

        # decide based on which handle was clicked
        if resize_corner == "br":
            x1, y1 = mx, my
        elif resize_corner == "bl":
            x0, y1 = mx, my
        elif resize_corner == "tr":
            x1, y0 = mx, my
        elif resize_corner == "tl":
            x0, y0 = mx, my

        if x1 <= x0: x1 = x0 + 1
        if y1 <= y0: y1 = y0 + 1

        width = x1 - x0
        height = y1 - y0

        r["col"], r["row"] = x0, y0
        r["colspan"], r["rowspan"] = width, height

        r["patch"].set_xy((x0, y0))
        r["patch"].set_width(width)
        r["patch"].set_height(height)

        r["label"].set_position((x0 + width/2, y0 + height/2))

        draw_handles(r)
        fig.canvas.draw_idle()

    # --------------------------------------------------
    # DRAWING NEW RECTANGLE
    # --------------------------------------------------
    if current_rect is None:
        return

    if start is None:
        return

    x0, y0 = start

    x1 = int(event.xdata)
    y1 = int(event.ydata)

    xmin = min(x0, x1)
    ymin = min(y0, y1)

    width = abs(x1 - x0) + 1
    height = abs(y1 - y0) + 1

    current_rect.set_xy((xmin, ymin))
    current_rect.set_width(width)
    current_rect.set_height(height)

    fig.canvas.draw_idle()


# ==========================================================
# MOUSE RELEASE
# ==========================================================

def on_release(event):

    global start
    global current_rect
    global resize_corner
    global selected_idx

    # Finish resize
    if resize_mode:

        resize_corner = None
        selected_idx = None
        return

    if start is None:
        return

    if current_rect is None:
        return

    x, y = current_rect.get_xy()

    w = int(current_rect.get_width())
    h = int(current_rect.get_height())

    label = ax.text(
        x + w / 2,
        y + h / 2,
        f"ax{len(rectangles)+1}",
        ha="center",
        va="center",
        fontsize=9,
        weight="bold",
        color=current_rect.get_edgecolor(),
    )

    rectangles.append(
        {
            "row": int(y),
            "col": int(x),
            "rowspan": h,
            "colspan": w,
            "projection": projection_mode,
            "patch": current_rect,
            "label": label,
        }
    )

    start = None
    current_rect = None

    fig.canvas.draw_idle()

    # ==========================================================
# EXPORT FUNCTION
# ==========================================================

def export(event=None):

    print("\nfrom matplotlib.gridspec import GridSpec")
    print("import matplotlib.pyplot as plt")
    print("from mpl_toolkits.mplot3d import Axes3D\n")

    print("fig = plt.figure(figsize=(10,10))")
    print("gs = GridSpec(100,100,figure=fig)\n")

    for i, r in enumerate(rectangles, 1):

        proj = r["projection"]

        if proj == "polar":
            print(
                f"ax{i} = fig.add_subplot("
                f"gs[{r['row']}:{r['row']+r['rowspan']}, "
                f"{r['col']}:{r['col']+r['colspan']}], "
                f"projection='polar')"
            )

        elif proj == "3d":
            print(
                f"ax{i} = fig.add_subplot("
                f"gs[{r['row']}:{r['row']+r['rowspan']}, "
                f"{r['col']}:{r['col']+r['colspan']}], "
                f"projection='3d')"
            )

        else:
            print(
                f"ax{i} = fig.add_subplot("
                f"gs[{r['row']}:{r['row']+r['rowspan']}, "
                f"{r['col']}:{r['col']+r['colspan']}])"
            )

    print("\nplt.show()")


btn_export.on_clicked(export)

# ==========================================================
# PREVIEW FUNCTION
# ==========================================================

def preview(event=None):

    fig2 = plt.figure(figsize=(9,9))
    gs = fig2.add_gridspec(GRID_ROWS, GRID_COLS)

    for r in rectangles:

        proj = r["projection"]

        axp = fig2.add_subplot(
            gs[
                r["row"]:r["row"]+r["rowspan"],
                r["col"]:r["col"]+r["colspan"]
            ],
            projection=proj if proj in ["polar", "3d"] else None
        )

        if proj == "polar":
            axp.plot(xsine,ysine)

        elif proj == "3d":
            axp.plot(xsine,xsine,ysine)

        else:
            axp.plot(xsine,ysine)

    plt.show()


btn_preview.on_clicked(preview)

# ==========================================================
# EVENT CONNECTIONS
# ==========================================================

fig.canvas.mpl_connect("button_press_event", on_press)
fig.canvas.mpl_connect("motion_notify_event", on_motion)
fig.canvas.mpl_connect("button_release_event", on_release)

# default active mode highlight
btn_cart.ax.set_facecolor("lightgreen")

# ==========================================================
# SHOW
# ==========================================================

fig.suptitle(
    "Scientific Figure Layout Designer",
    y=0.91,
    fontsize=12
)

plt.show()